In [0]:
from pyspark.sql import functions as F

### Create a big table containing 200 thousands rows

In [0]:
df_raw = spark.range(0,20000000) \
    .withColumn("groupBy_key",F.pmod(F.col("id"),F.lit(5))) \
    .withColumn("value",F.rand(seed=66))

In [0]:
df_raw.distinct().groupby("groupBy_key").count().show()

Analyzed the PySpark DataFrame operation that processes 20M in-memory rows with deduplication and grouping.

Summary

The query generates 20M rows in-memory, applies deduplication across all columns, then groups by a modulo key. The aggregation step consumes 124ms (77% of operator execution time) and 48MB of memory to produce 5 groups.

Details and metrics

The operation generates 20M unique IDs using Range, adds a grouping key (ID modulo 5) and random values
The deduplication step processes all 20M rows before aggregation; since the source contains unique IDs and random values with a fixed seed, duplicate rows are extremely unlikely
The aggregation phase reduces 20M rows down to 5 groups
Total execution time is 510ms (509ms compilation, 1ms execution); no external data is read
The .distinct() operation processes the full dataset but likely finds no duplicates to remove, given that spark.range() produces unique IDs. If you can confirm the dataset contains no duplicate rows (where id, groupBy_key, and value all match), removing .distinct() would eliminate unnecessary deduplication overhead and allow the groupby to process the data directly.
